In [1]:
print("Kernel works")

Kernel works


In [2]:
import pandas as pd
from rdkit import Chem, RDLogger

# Hide RDKit informational and warning messages
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/df_chem.csv')
print(df.columns)
print("Dataset shape:", df.shape)
print("Missing MOFID:", df["mofid"].isna().sum())
print("Unique MOFID:", df["mofid"].nunique())


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')
Dataset shape: (27706, 11)
Missing MOFID: 0
Unique MOFID: 24958


In [3]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


---
we want to answer three questions:

- How many extra occurrences are caused by repeated MOFIDs?
- How many different MOFID strings appear more than once?
- How many times can one MOFID occur?

Step 1 — Count occurrence of each MOFID.


In [4]:
mofid_counts = df["mofid"].value_counts()
# Keep only MOFIDs that appear more than once
repeated_mofids = mofid_counts[mofid_counts > 1]

print("Total rows:", len(df))
print("Unique MOFIDs:", df["mofid"].nunique())
print("Repeated occurrences:", df["mofid"].duplicated().sum())

print(mofid_counts.head())


Total rows: 27706
Unique MOFIDs: 24958
Repeated occurrences: 2748
mofid
* MOFid-v1.NA.NA                                                                         1259
* MOFid-v1.NA.NAno_mof                                                                    495
N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERROR.cat0                                     39
N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERROR.cat0                                     32
[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21
Name: count, dtype: int64


| Output                                | Meaning                                                                                                                                             |
| ------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Total rows: 27,706**                |  dataset contains 27,706 MOF records with a non-null value in the `mofid` column.                                                               |
| **Unique MOFIDs: 24,958**             | There are 24,958 different `mofid` strings among those 27,706 rows.                                                                                 |
| **Extra repeated occurrences: 2,748** | After keeping the first occurrence of every MOFID, there are 2,748 additional occurrences of already-seen MOFID strings. This is `27,706 − 24,958`. |
| **Unique MOFIDs that repeat: 616**    | There are 616 different MOFID strings that occur at least twice.                                                                                    |
| **Maximum occurrence: 1,259**         | The most frequently occurring MOFID string appears in 1,259 rows.                                                                                   |


Total rows: 27706
Unique MOFIDs: 24958
Extra repeated occurrences: 2748
Unique MOFIDs that repeat: 616 `number of repeated MOFID types`
Maximum occurrence of one MOFID: 1259 `highest frequency of one MOFID type`
mofid

---


`* MOFid-v1.NA.NA    1259`
means this exact string occurs in 1,259 rows. NA indicates that a normal MOFID representation was not available/generated, so these are not 1,259 copies of the same chemical structure.

---
*MOFid-v1.NA.NAno_mof   495*
means this exact status-like string occurs 495 times. Again, this is not a normal chemical MOFID.

---

*N#N.[O-]C(=O)C#CC(=O)[O-].[Zn][Zn] MOFid-v1.ERRORcat0  39*

means the exact same MOFID string occurs 39 times. Unlike the NA entries, it contains chemical fragments (N#N, linker, Zn) but its MOFID metadata contains ERROR.

---

*N#N.[Cu][Cu].[O-]C(=O)C#CC(=O)[O-] MOFid-v1.ERRORcat0   32*
is another specific chemical representation occurring 32 times, also carrying an ERROR status.

---
*[O-]C(=O)C#CC(=O)[O-].[O-]C(=O)C=CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1      21*
occurs 21 times. This looks different because pcu is a topology designation rather than an NA or ERROR marker.

*The key distinction is:*
27,706 rows ≠ 27,706 unique MOFIDs. There are 24,958 unique MOFID strings, and the repetition is strongly influenced by placeholder/error values such as the 1,259 NA records.

removing both
* MOFid-v1.NA.NA
          ^^^^^^^^^^

* MOFid-v1.NA.NAno_mof
          ^^^^^^^^^^

In [5]:
# removing both
# MOFid-v1.NA.NA
#  MOFid-v1.NA.NAno_mof
df = df[
    ~df["mofid"].str.contains("MOFid-v1.NA", na=False)
].copy()

print("Remaining rows:", len(df))
#Then verify:
print(df.shape)
print(df["mofid"].str.contains("MOFid-v1.NA", na=False).sum())


Remaining rows: 25952
(25952, 11)
0


In [6]:
# quantify only the ERROR records:
error_count = df["mofid"].str.contains(
    "MOFid-v1.ERROR", na=False
).sum()

print("ERROR-type MOFIDs:", error_count)

# take Take one ERROR MOFID to check it works with rdkit
error_example = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"].iloc[0]

print(error_example)

# extract it :
chemical_smiles = error_example.split(" ")[0]
print(chemical_smiles)

# Test with RDKIT 
mol = Chem.MolFromSmiles(chemical_smiles)
print(mol)

ERROR-type MOFIDs: 1691
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.ERROR
N#Cc1ccc2c(c1)C1([CH]C(=C2C(=O)[O-])C#N)C(=O)ON1[C]C1=C[C]2[C]N=N[C]c3c(C#N)cc4c(c3C#N)c3c(C#N)c(C(=O)[O-])c(-c5cccc([C]N=N[C]c6c(c(-c7cc8C(=O)OC1C(=C2)c8cc7C(=O)[O-])ccc6)C#N)c5)cc3c(=O)oc4=[N].[Zn][O]([Zn])([Zn])[Zn]
None


In [7]:
# Now test all 1,691 ERROR records with RDkit : 
error_mofids = df[
    df["mofid"].str.contains("MOFid-v1.ERROR", na=False)
]["mofid"]

parsed = 0
failed = 0

for mofid in error_mofids:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
    else:
        parsed += 1

print("Total ERROR records:", len(error_mofids))
print("Successfully parsed:", parsed)
print("Failed to parse:", failed)

Total ERROR records: 1691
Successfully parsed: 1628
Failed to parse: 63


---

*27,706 non-null MOFID rows → remove 1,754 NA placeholders → 25,952 rows → 1,691 ERROR-labelled records → 1,628 parse successfully and only 63 fail.*

---




### Can RDKit parse the chemical representation for all 25,952 remaining records?

In [8]:
print("Dataset shape:", df.shape)
# first look at one MOFID from  dataset:
mofid = df["mofid"].iloc[0]

print(mofid)

Dataset shape: (25952, 11)
[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0


---
I has two part chemical representation and MOFid metatdata:  `MOFid-v1.pcu.cat0`

---

In [9]:
# .split(" ") separates the MOFID string at the space.
# [0] selects the chemical representation before the MOFID metadata.
chemical_smiles = mofid.split(" ")[0]

print(chemical_smiles)

from rdkit import Chem

mol = Chem.MolFromSmiles(chemical_smiles)

print(mol)

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]
None


----
whether we can isolate the linker from your MOFID and process that linker with RDKit.

----

In [10]:
fragments = chemical_smiles.split(".")

print(fragments)
fragment = Chem.MolFromSmiles(fragments[0])

print(fragment)

['[O-]C(=O)c1ccc(cc1)C(=O)[O-]', '[Zn][O]([Zn])([Zn])[Zn]']


---
<rdkit.Chem.rdchem.Mol object at 0x75c18c10e8f0>

RDKit successfully parsed the first fragment.

---

In [11]:
# Check whole dataset with RDKit
failed_mofids = []
parsed = 0  # number RDKit successfully reads
failed = 0  # number RDKit cannot read
for mofid in df["mofid"]:
    chemical_smiles = mofid.split(" ")[0]
    mol = Chem.MolFromSmiles(chemical_smiles)

    if mol is None:
        failed += 1
        failed_mofids.append(mofid)
    else:
        parsed += 1

print("Successfully parsed:", parsed)
print("Failed to parse:", failed)


Successfully parsed: 12940
Failed to parse: 13012


---
Can standard RDKit parse all 25,952 complete chemical representations?

No. It parses 12,940, while 13,012 fail.

Next Step: should be only to collect the failed MOFIDs so we can inspect what causes these failures.

----

In [12]:
# collect failed mofid to understand about their features:
# We check a small batch of dataset
for mofid in failed_mofids[:10]:
    print(mofid)
    print()

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C(=O)C(=O)[O-].[O-]C(=O)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.UNKNOWN.cat0

[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)cc(c(c2)C(=O)[O-])C.[O-]C(=O)c1ccc2c(c1C)c(C)c(c(c2C)C(=O)[O-])C.[Zn][O]([Zn])([Zn])[Zn] MOFid-v1.pcu.cat1

-----

This supports hypothesis that RDKit may be rejecting the Zn–O coordination fragment, while the organic linker itself may still be readable.

-----

In [13]:
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?
# How many of the 13,012 failed MOFIDs contain this exact Zn–O fragment?

zn_o_cluster = "[Zn][O]([Zn])([Zn])[Zn]"

with_zn_o_cluster = 0
without_zn_o_cluster = 0

for mofid in failed_mofids:
    if zn_o_cluster in mofid:
        with_zn_o_cluster += 1
    else:
        without_zn_o_cluster += 1

print(
    f"With Zn-O cluster: {with_zn_o_cluster}, "
    f"Without Zn-O cluster: {without_zn_o_cluster}"
)

With Zn-O cluster: 12284, Without Zn-O cluster: 728


In [14]:
without_zn_o_cluster = []
for mofid in failed_mofids:
    if zn_o_cluster not in mofid:
        without_zn_o_cluster.append(mofid)

for mofid in without_zn_o_cluster[:10]:
    print(mofid)
    print() 

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1cc(O)c(cc1O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

OC1=[N]=C(C(=N[CH]1)O)O.[O-]C(=O)c1ccc(c(c1)O)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat1

CCOC1=[N]=C(C=N[CH]1)OCC.CCOc1cc(cc(c1C(=O)[O-])OCC)C(=O)[O-].[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

CCOC1=[N]=C(C(=N[CH]1)OCC)OCC.CCOc1cc(C(=O)[O-])c(cc1C(=O)[O-])OCC.[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

[O-]C(=O)c1ccc(cc1)C(=O)[O-].CCCOC1=[N]=C(C(=N[CH]1)OCCC)OCCC.[O-]C(=O)c1ccc(cc1OCCC)C(=O)[O-].[Zn][Zn] MOFid-v1.pcu.cat0

COC1=[N]=C(OC)C=C([CH]1)c1ccncc1OC.COc1cc(C(=O)[O-]

In [15]:
# get overall overview of data :
error_count = 0
pcu_cat0_count = 0
pcu_cat1_count = 0
n2_count = 0

for mofid in failed_mofids:
    if "ERROR" in mofid:
        error_count += 1
    if "pcu.cat0" in mofid:
        pcu_cat0_count += 1
    if "pcu.cat1" in mofid:
        pcu_cat1_count += 1
    if "N#N" in mofid:
        n2_count += 1

print("ERROR:", error_count)
print("pcu.cat0:", pcu_cat0_count)
print("pcu.cat1:", pcu_cat1_count)
print("Contains N#N:", n2_count)

ERROR: 63
pcu.cat0: 7210
pcu.cat1: 3659
Contains N#N: 6


| Pattern        | Failed MOFIDs |
| -------------- | ------------: |
| `ERROR`        |            63 |
| `pcu.cat0`     |         7,210 |
| `pcu.cat1`     |         3,659 |
| Contains `N#N` |             6 |


In [16]:
# fragment analysis and Linker extraction:
# Step 1 :Extract the chemical portion of each MOFID by removing the MOFid-v1... metadata.
chemical_representations = []

for mofid in df["mofid"]:
    chemical_smiles = mofid.split("MOFid-v1")[0].strip() # space and mofide_v1“For every MOFID, remove the MOFID metadata and store only its chemical representation.”
    chemical_representations.append(chemical_smiles)

for chemical in chemical_representations[:5]:
    print(chemical)
    print()


[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1C(=O)[O-])C(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]

CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=O)C#CC(=O)[O-].[Zn][O]([Zn])([Zn])[Zn]



In [17]:
df["chemical_representation"] = chemical_representations
print(df.columns)
df_fragments = df.copy()

Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg', 'chemical_representation'],
      dtype='object')


In [18]:
# Metal Discconector
from rdkit.Chem.MolStandardize import rdMolStandardize
chemical = df["chemical_representation"].iloc[0]
# Create RDKit molecule without standard sanitization
mof_mol = Chem.MolFromSmiles(chemical, sanitize=False)
mof_mol.UpdatePropertyCache(strict=False)
print(mof_mol)
# Initialize MetalDisconnector
disconnector = rdMolStandardize.MetalDisconnector()
# Disconnect metal-nonmetal bonds
disconnected_mol = disconnector.Disconnect(mof_mol)


In [19]:
# Split the disconnected MOF into individual molecular fragments
fragments = Chem.GetMolFrags(disconnected_mol, asMols=True)

# Print the total number of fragments obtained
print(len(fragments))

# Convert each RDKit fragment back to a SMILES string and display it
for fragment in fragments:
    print(Chem.MolToSmiles(fragment))


6
O=C([O-])c1ccc(C(=O)[O-])cc1
[Zn+]
[O-2]
[Zn+]
[Zn]
[Zn]


In [20]:
# Define the atomic numbers that correspond to metals
metal_atomic_numbers = (
    list(range(3, 5)) +
    list(range(11, 14)) +
    list(range(19, 32)) +
    list(range(37, 51)) +
    list(range(55, 85)) +
    list(range(87, 113))
)
# Define SMARTS for unwanted inorganic nodes/solvents
inorganic_blacklist = [
    Chem.MolFromSmarts('[O-2]'),       # Oxide ion bridges
    Chem.MolFromSmarts('N#N'),         # Nitrogen gas
    Chem.MolFromSmarts('[O;H2]'),       # Water
    Chem.MolFromSmarts('[O-]S(=O)(=O)[O-]') # Sulfate (if applicable)
]
# Create an empty list for fragments that are not metal atoms
non_metal_fragments = []

# Examine each fragment from the first MOF
for frag in fragments:

    # If the fragment contains exactly one atom
    if frag.GetNumAtoms() == 1:

        # Get that atom
        atom = frag.GetAtomWithIdx(0)

        # If that atom is a metal, skip it
        if atom.GetAtomicNum() in metal_atomic_numbers:
            continue
      # Check if the fragment matches any blacklisted substructure completely
    is_inorganic = False
    for pattern in inorganic_blacklist:
        if frag.HasSubstructMatch(pattern):
            # Ensure the match isn't just a part of a larger organic molecule
            if frag.GetNumHeavyAtoms() == pattern.GetNumHeavyAtoms():
                is_inorganic = True
                break
                
    if not is_inorganic:
    # Keep everything that was not removed as a metal
        non_metal_fragments.append(frag)

# Display the fragments remaining after metal removal
for frag in non_metal_fragments:
    print(Chem.MolToSmiles(frag))

O=C([O-])c1ccc(C(=O)[O-])cc1


In [ ]:
# Define the atomic numbers that correspond to metals
metal_atomic_numbers = (
    list(range(3, 5)) +
    list(range(11, 14)) +
    list(range(19, 32)) +
    list(range(37, 51)) +
    list(range(55, 85)) +
    list(range(87, 113))
)
# Define SMARTS for unwanted inorganic nodes/solvents
# Substructure patterns
inorganic_blacklist = [
    Chem.MolFromSmarts("[O-2]"),
    Chem.MolFromSmarts("N#N"),
    Chem.MolFromSmarts("[O;H2]"),
    Chem.MolFromSmarts("[O-]S(=O)(=O)[O-]")
]
# Exact standalone fragments
exact_fragment_blacklist = {
    "[Zn][Zn]",
    "[Cu][Cu]",
    "[O]"
}    
def extract_linkers(chemical):
    # Read the MOF representation without sanitizing
    lig_mol = Chem.MolFromSmiles(chemical, sanitize=False)
    if lig_mol is None:
        raise ValueError(
            "RDKit could not create a molecule from the chemical representation"
        )
    lig_mol.UpdatePropertyCache(strict=False)

    # Disconnect metal-ligand bonds
    disconnector = rdMolStandardize.MetalDisconnector()
    disconnected_mol = disconnector.Disconnect(lig_mol)

    # Split the disconnected structure into individual fragments
    fragments = Chem.GetMolFrags(
        disconnected_mol,
        asMols=True,
        sanitizeFrags=False
    )
    linker_smiles = []
    invalid_fragments = []
    for frag in fragments:
        # Remove isolated metal atoms
        if frag.GetNumAtoms() == 1:
            atom = frag.GetAtomWithIdx(0)
            if atom.GetAtomicNum() in metal_atomic_numbers:
                continue
        # Remove known small inorganic fragments
        is_inorganic = any(
            frag.HasSubstructMatch(pattern)
            and frag.GetNumHeavyAtoms() == pattern.GetNumHeavyAtoms()
            for pattern in inorganic_blacklist
        )
        if is_inorganic:
            continue
        # Convert this fragment to an exact SMILES
        fragment_smiles = Chem.MolToSmiles(frag)
        # Remove exact unwanted standalone fragments
        if fragment_smiles in exact_fragment_blacklist:
            continue
        # Now sanitize ONLY this candidate fragment
        try:
            frag.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(frag)
            # Create the final sanitized SMILES
            fragment_smiles = Chem.MolToSmiles(frag)
            # Remove exact unwanted fragments
            if fragment_smiles in exact_fragment_blacklist:
                continue
            linker_smiles.append(fragment_smiles)
        except Exception as error:
            invalid_fragments.append({
            "fragment": Chem.MolToSmiles(frag),
            "error": str(error)
        })
    return linker_smiles, invalid_fragments

In [23]:
test_result = extract_linkers(
    df_fragments["chemical_representation"].iloc[0]
)

print(test_result)

(['O=C([O-])c1ccc(C(=O)[O-])cc1'], [])


In [24]:
# Create an empty list to store valid linkers for every MOF
all_linkers = []

# Create an empty list to store invalid fragments for every MOF
all_invalid_fragments = []

# Record rows where the complete extraction function fails
failed_linker_extraction = []


# Loop through every chemical representation
for index, chemical in df_fragments[
    "chemical_representation"
].items():

    try:
        # The function returns two results
        linkers, invalid_fragments = extract_linkers(
            chemical
        )

        # Store the valid linker list
        all_linkers.append(linkers)

        # Store the invalid-fragment list
        all_invalid_fragments.append(
            invalid_fragments
        )

    except Exception as error:
        # Add empty lists to preserve the row alignment
        all_linkers.append([])
        all_invalid_fragments.append([])

        # Save information about the failed row
        failed_linker_extraction.append({
            "index": index,
            "chemical_representation": chemical,
            "error": str(error)
        })


# Add the valid linker lists as a new column
df_fragments["linker_smiles"] = all_linkers

# Add the invalid fragment information as another column
df_fragments["invalid_fragments"] = (
    all_invalid_fragments
)


# Check the results
print(
    "Rows processed:",
    len(df_fragments)
)

print(
    "Complete extraction failures:",
    len(failed_linker_extraction)
)

print(
    df_fragments[
        [
            "chemical_representation",
            "linker_smiles",
            "invalid_fragments"
        ]
    ].head()
)

Rows processed: 25952
Complete extraction failures: 4
                             chemical_representation  \
0  [O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...   
1  [O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...   
2  [O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...   
3  COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...   
4  CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...   

                                       linker_smiles invalid_fragments  
0                     [O=C([O-])c1ccc(C(=O)[O-])cc1]                []  
1                     [O=C([O-])c1ccc(C(=O)[O-])cc1]                []  
2              [O=C([O-])c1cc(F)c(C(=O)[O-])c(F)c1F]                []  
3  [COc1cc(C(=O)[O-])cc(OC)c1C(=O)[O-], COc1cc(C(...                []  
4  [CCc1cc(C(=O)[O-])c(CC)c(CC)c1C(=O)[O-], O=C([...                []  


In [25]:
# Select only MOFs where no linker was extracted
failed_linkers_df = df_fragments[
    df_fragments["linker_smiles"].apply(len) == 0
].copy()

# Check the number of rows without extracted linkers
print(failed_linkers_df.shape)

# Inspect their chemical representations
display(
    failed_linkers_df[
        [
            "mofid",
            "chemical_representation",
            "linker_smiles"
        ]
    ].head(20)
)

(23, 14)


,mofid,chemical_representation,linker_smiles
3023,[O-]C(=O)C1=C[C]=C(C=C1)c1ccc(cc1Cl)c1ccc(c(c1...,[O-]C(=O)C1=C[C]=C(C=C1)c1ccc(cc1Cl)c1ccc(c(c1...,[]
5616,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,[]
8121,CCCOC1=[N]=C(C(=O)N=C1)OCCC.[Zn][Zn] MOFid-v1....,CCCOC1=[N]=C(C(=O)N=C1)OCCC.[Zn][Zn],[]
10568,N#N.[Zn][Zn] MOFid-v1.ERROR,N#N.[Zn][Zn],[]
10570,N#N.[Zn][Zn] MOFid-v1.UNKNOWN,N#N.[Zn][Zn],[]
10585,N#N.[O-]C(=O)[C]1[NH2]c2cccc3-c4cc(c(-c5cc([NH...,N#N.[O-]C(=O)[C]1[NH2]c2cccc3-c4cc(c(-c5cc([NH...,[]
10760,COC1=[N]=C(C(=O)N=C1)OC.[Zn][Zn] MOFid-v1.pcu....,COC1=[N]=C(C(=O)N=C1)OC.[Zn][Zn],[]
11070,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,[]
11782,CCCOC1=[N]=C(OCCC)C(=C([CH]1)c1ccnc(c1)OCCC)OC...,CCCOC1=[N]=C(OCCC)C(=C([CH]1)c1ccnc(c1)OCCC)OC...,[]
13101,NC1=NC(=C2C(=[C]1)c1c(N)cncc1[NH2][NH2]2)N.[Zn...,NC1=NC(=C2C(=[C]1)c1c(N)cncc1[NH2][NH2]2)N.[Zn...,[]


In [26]:
# Collect the indexes of the four complete extraction failures
failed_indexes = {
    failure["index"]
    for failure in failed_linker_extraction
}

# Label the reason for each empty-linker row
failed_linkers_df["reason"] = [
    "complete extraction failure"
    if index in failed_indexes
    else "all fragments excluded"
    for index in failed_linkers_df.index
]

# Count each reason
print(
    failed_linkers_df["reason"]
    .value_counts()
)

reason
all fragments excluded         19
complete extraction failure     4
Name: count, dtype: int64


The total number excluded from the chemistry dataset is 23, not 4.

4: complete RDKit extraction failures
19: extraction succeeded, but no organic linker remained after removing inorganic/metal fragments
Total excluded: 4+19=23

In [27]:
# Create a dictionary connecting each failed row index
# to its complete extraction error message
error_by_index = {
    failure["index"]: failure["error"]
    for failure in failed_linker_extraction
}


# Create a DataFrame containing all 23 excluded MOFs
excluded_linker_df = failed_linkers_df[
    [
        "mofid",
        "chemical_representation",
        "linker_smiles",
        "invalid_fragments",
        "reason"
    ]
].copy()


# Add the RDKit error message for the four complete failures
# The other 19 rows will contain NaN because they had no error
excluded_linker_df["error"] = (
    excluded_linker_df.index.map(error_by_index)
)


# Check the shape and available columns
print("Excluded rows:", excluded_linker_df.shape)
print(excluded_linker_df.columns)


# Display the first records
display(excluded_linker_df.head(23))

Excluded rows: (23, 6)
Index(['mofid', 'chemical_representation', 'linker_smiles',
       'invalid_fragments', 'reason', 'error'],
      dtype='object')


,mofid,chemical_representation,linker_smiles,invalid_fragments,reason,error
3023,[O-]C(=O)C1=C[C]=C(C=C1)c1ccc(cc1Cl)c1ccc(c(c1...,[O-]C(=O)C1=C[C]=C(C=C1)c1ccc(cc1Cl)c1ccc(c(c1...,[],[{'fragment': 'O=C([O-])C1=CC=C(c2ccc(c3ccc(C(...,all fragments excluded,NaN
5616,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,[O-]C(=O)C1=C(Br)[C]=C(C(=C1)Br)C(=O)[O-].[O-]...,[],[],complete extraction failure,"Explicit valence for atom # 118 Br, 2, is grea..."
8121,CCCOC1=[N]=C(C(=O)N=C1)OCCC.[Zn][Zn] MOFid-v1....,CCCOC1=[N]=C(C(=O)N=C1)OCCC.[Zn][Zn],[],"[{'fragment': 'CCCOC1=N=C(OCCC)C(=O)N=C1', 'er...",all fragments excluded,NaN
10568,N#N.[Zn][Zn] MOFid-v1.ERROR,N#N.[Zn][Zn],[],[],all fragments excluded,NaN
10570,N#N.[Zn][Zn] MOFid-v1.UNKNOWN,N#N.[Zn][Zn],[],[],all fragments excluded,NaN
10585,N#N.[O-]C(=O)[C]1[NH2]c2cccc3-c4cc(c(-c5cc([NH...,N#N.[O-]C(=O)[C]1[NH2]c2cccc3-c4cc(c(-c5cc([NH...,[],[{'fragment': 'O=C([O-])C1[NH2]c2cccc(c2)-c2cc...,all fragments excluded,NaN
10760,COC1=[N]=C(C(=O)N=C1)OC.[Zn][Zn] MOFid-v1.pcu....,COC1=[N]=C(C(=O)N=C1)OC.[Zn][Zn],[],"[{'fragment': 'COC1=N=C(OC)C(=O)N=C1', 'error'...",all fragments excluded,NaN
11070,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,O=C1C#Cc2ccc3cc2[C]2[N]45[O]62[Zn]27895[NH]4N4...,[],[],complete extraction failure,"Explicit valence for atom # 12 O, 3, is greate..."
11782,CCCOC1=[N]=C(OCCC)C(=C([CH]1)c1ccnc(c1)OCCC)OC...,CCCOC1=[N]=C(OCCC)C(=C([CH]1)c1ccnc(c1)OCCC)OC...,[],[{'fragment': 'CCCOC1=N=C(OCCC)C(OCCC)=C(c2ccn...,all fragments excluded,NaN
13101,NC1=NC(=C2C(=[C]1)c1c(N)cncc1[NH2][NH2]2)N.[Zn...,NC1=NC(=C2C(=[C]1)c1c(N)cncc1[NH2][NH2]2)N.[Zn...,[],[{'fragment': 'NC1=NC(N)=C2[NH2][NH2]c3cncc(N)...,all fragments excluded,NaN


## Note: 



Valid organic linkers were retained for 25,929 of 25,952 MOFs (99.91%). A total of 23 structures were excluded from chemistry-descriptor generation:

4 structures produced unresolved RDKit explicit-valence errors after metal disconnection—three O-valence cases and one Br-valence case.
19 structures were processed successfully, but no valid organic linker remained after metal and inorganic fragments were removed.

No manual charge or bonding corrections were applied, avoiding unintended alteration of the reported chemical structures.

In [29]:
# Select the 23 MOFs that have no valid organic linker
failed_final_df = df_fragments[
    df_fragments["linker_smiles"].apply(len) == 0
].copy()

# Confirm the number of excluded rows
print("MOFs to exclude:", len(failed_final_df))

# Display their filenames
print(failed_final_df["filename"])

MOFs to exclude: 23
3023     hMOF-12739
5616       hMOF-151
8121     hMOF-17422
10568    hMOF-19759
10570    hMOF-19760
10585    hMOF-19779
10760    hMOF-19962
11070    hMOF-20326
11782    hMOF-21484
13101    hMOF-22905
14155    hMOF-23902
14191    hMOF-23939
14831    hMOF-24526
15464    hMOF-25105
17126    hMOF-26645
19114    hMOF-28530
21038    hMOF-30369
22034    hMOF-31424
24142      hMOF-368
24228     hMOF-3757
24617     hMOF-4107
27218     hMOF-6556
27393     hMOF-6716
Name: filename, dtype: object


In [31]:
# Create a new DataFrame containing only successfully processed MOFs
# Keep only MOFs with at least one valid organic linker
df_chemistry = df_fragments[
    df_fragments["linker_smiles"].apply(len) > 0
].copy()

# Check the new dataset size
print("Original Phase 2 dataset:", df_fragments.shape)
print("Final chemistry dataset:", df_chemistry.shape)

# Display the final column names
df_chemistry.columns

Original Phase 2 dataset: (25952, 14)
Final chemistry dataset: (25929, 14)


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg', 'chemical_representation', 'linker_smiles',
       'invalid_fragments'],
      dtype='object')

In [32]:
print(df_chemistry["linker_smiles"].tail(10))
print(df_chemistry["linker_smiles"].head(10))
print(df_chemistry["linker_smiles"].iloc[0])
print(type(df_chemistry["linker_smiles"].iloc[0]))

27696    [O=C([O-])C#CC(=O)[O-], O=C([O-])C=CC(F)=C(F)C...
27697    [O=C([O-])C#CC(=O)[O-], O=C([O-])C=CC(F)=C(F)C...
27698    [O=C([O-])C#CC(=O)[O-], O=C([O-])C=CC(Cl)=CC(=...
27699    [O=C([O-])C#CC(=O)[O-], O=C([O-])C=CC(Cl)=CC(=...
27700    [O=C([O-])c1cc(F)c(C(=O)[O-])c(F)c1F, O=C([O-]...
27701                      [Nc1cc(C(=O)[O-])ccc1C(=O)[O-]]
27702    [Cc1cc(C(=O)[O-])c2c(c1C(=O)[O-])C(C)C2(C)C, C...
27703    [O=C([O-])C#CC(=O)[O-], O=C([O-])C(Cl)=CC(Cl)=...
27704    [O=C([O-])C#CC(=O)[O-], O=C([O-])C(Cl)=CC(Cl)=...
27705                 [O=C([O-])[C]=CC(Cl)=C(Cl)C(=O)[O-]]
Name: linker_smiles, dtype: object
0                       [O=C([O-])c1ccc(C(=O)[O-])cc1]
1                       [O=C([O-])c1ccc(C(=O)[O-])cc1]
2                [O=C([O-])c1cc(F)c(C(=O)[O-])c(F)c1F]
3    [COc1cc(C(=O)[O-])cc(OC)c1C(=O)[O-], COc1cc(C(...
4    [CCc1cc(C(=O)[O-])c(CC)c(CC)c1C(=O)[O-], O=C([...
5    [O=C([O-])c1cc(Br)c2cc(C(=O)[O-])ccc2c1, O=C([...
6    [O=C([O-])C(=O)[O-], Cc1c(C(=O)[O-])ccc2

In [33]:
unwanted = {"[Zn][Zn]", "[Cu][Cu]", "[O]"}

remaining = df_chemistry["linker_smiles"].apply(
    lambda value: [smiles for smiles in value if smiles in unwanted]
)

remaining = remaining[remaining.str.len() > 0]

print("Rows still containing unwanted fragments:", len(remaining))
print()
print(
    remaining
    .explode()
    .value_counts()
)

Rows still containing unwanted fragments: 0

Series([], Name: count, dtype: int64)


---

#### Calculate RDKit linker descriptors will add numerical columns such as:
```text
- linker_molecular_weight
- linker_tpsa
- linker_logp
- linker_h_bond_donors
- linker_h_bond_acceptors
- linker_rotatable_bonds
- linker_aromatic_fraction

In [34]:
# Import RDKit tools for reading SMILES and calculating descriptors
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

first_linker = df_chemistry["linker_smiles"].iloc[0][0]

mol = Chem.MolFromSmiles(first_linker)

print("Linker:", first_linker)
print("Molecular weight:", Descriptors.MolWt(mol))

Linker: O=C([O-])c1ccc(C(=O)[O-])cc1
Molecular weight: 164.11599999999999


In [35]:
# Create a function that calculates descriptors for one linker SMILES
def calculate_linker_descriptors(smiles):

    # Convert the linker SMILES into an RDKit molecule
    mol = Chem.MolFromSmiles(smiles)

    # Return None if RDKit cannot read the linker
    if mol is None:
        return None

    # Count the number of non-hydrogen atoms
    heavy_atoms = mol.GetNumHeavyAtoms()

    # Count the atoms that are part of aromatic rings
    aromatic_atoms = sum(
        atom.GetIsAromatic()
        for atom in mol.GetAtoms()
    )

    # Avoid division by zero when calculating aromatic fraction
    if heavy_atoms > 0:
        aromatic_fraction = aromatic_atoms / heavy_atoms
    else:
        aromatic_fraction = 0

    # Return the calculated descriptors as a dictionary
    return {
        "molecular_weight": Descriptors.MolWt(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "logp": Descriptors.MolLogP(mol),
        "h_bond_donors": rdMolDescriptors.CalcNumHBD(mol),
        "h_bond_acceptors": rdMolDescriptors.CalcNumHBA(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "ring_count": rdMolDescriptors.CalcNumRings(mol),
        "formal_charge": Chem.GetFormalCharge(mol),
        "aromatic_fraction": aromatic_fraction
    }

# Select the first linker from the first row
first_linker = df_chemistry["linker_smiles"].iloc[0][0]

# Calculate descriptors for this linker
first_descriptor_result = calculate_linker_descriptors(first_linker)

print("Linker:", first_linker)
print("Descriptors:", first_descriptor_result)

Linker: O=C([O-])c1ccc(C(=O)[O-])cc1
Descriptors: {'molecular_weight': 164.11599999999999, 'tpsa': 80.25999999999999, 'logp': -1.5864000000000003, 'h_bond_donors': 0, 'h_bond_acceptors': 4, 'rotatable_bonds': 2, 'ring_count': 1, 'formal_charge': -2, 'aromatic_fraction': 0.5}


| Descriptor       |       Result | Meaning                                                                                                                                                |
| ---------------- | -----------: | ------------------------------------------------------------------------------------------------------------------------------------------------------ |
| Molecular weight | 164.12 g/mol | The total atomic mass of the linker. Larger linkers generally have higher molecular weight.                                                            |
| TPSA             |     80.26 Å² | Topological polar surface area. This relatively high value indicates several polar oxygen atoms that may interact with CO₂.                            |
| LogP             |        −1.59 | Estimated hydrophobicity. The negative value indicates a strongly polar/hydrophilic linker. Because this linker is charged, interpret LogP cautiously. |
| H-bond donors    |            0 | The linker has no groups that donate hydrogen bonds. Its carboxylate groups are deprotonated, so they do not contain O–H bonds.                        |
| H-bond acceptors |            4 | Four atoms can accept hydrogen bonds. These are likely the four oxygen atoms in the two carboxylate groups.                                            |
| Rotatable bonds  |            2 | Two bonds can rotate. These are probably the bonds connecting the two carboxylate groups                                                               |

For CO₂ model:

- tpsa, h_bond_acceptors, and formal_charge describe linker polarity and possible CO₂ interactions.
- molecular_weight, ring_count, rotatable_bonds, and aromatic_fraction describe linker size, rigidity, and structure.
- logp provides another polarity-related descriptor, but it should be interpreted carefully for charged MOF linkers.

In [ ]:
print(df_chemistry['linker_smiles'])
# Save the final linker-extracted dataset
df_chemistry.to_csv(
    "/home/susan/mof-co2-adsorption/data/processed/hmof_linker_extracted.csv",
    index=False
)